# Twin-2K-500 — concise EDA

Four checks: **participant count, question types, A/B differences, and target leakage**.

Set `TWIN_DATA_ROOT` to your dataset folder and run all cells. Dependencies: `pandas`, `pyarrow`, `ipython`. All code is included. Metadata checks cover all participants; answer inspection is limited to training participant **574**. No test scoring or training runs here.

Source: [dataset and definitions, pinned revision f883165](https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500/tree/f883165a3026fde855dfd448e0cd16443ab257b6).

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import display, Markdown
import hashlib, json, os, random
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds

candidates = [Path(os.environ.get('TWIN_DATA_ROOT', 'twin_eda_data')),
              Path.cwd().parent/'twin_eda_data', Path('data/Twin-2K-500'),
              Path('/root/autodl-tmp/twin_qwen/data/Twin-2K-500')]
ROOT = next((p.resolve() for p in candidates if
    (p/'question_catalog_and_human_response_csv/question_catalog.json').is_file()), None)
assert ROOT, 'Set TWIN_DATA_ROOT to your downloaded Twin-2K-500 folder.'
paths = sorted((ROOT/'wave_split/chunks').glob('*.parquet'))
waves = ds.dataset([str(p) for p in paths], format='parquet')
catalog = json.loads((ROOT/'question_catalog_and_human_response_csv/question_catalog.json').read_text())
FIELDS = {'A': 'wave1_3_persona_json', 'B': 'wave4_Q_wave1_3_A'}

def decode(value):
    return json.loads(value) if isinstance(value, str) else value

def strip_answers(value):
    if isinstance(value, dict):
        return {k: strip_answers(v) for k, v in value.items()
                if k not in {'Answers', 'Answer', 'SelectedByPosition', 'SelectedText'}}
    return [strip_answers(v) for v in value] if isinstance(value, list) else value

def questions(value):
    if isinstance(value, dict):
        if 'QuestionID' in value and 'QuestionType' in value:
            yield value
        else:
            for k, child in value.items():
                if k not in {'Answers', 'Answer'}: yield from questions(child)
    elif isinstance(value, list):
        for child in value: yield from questions(child)

def blocks(value):
    if isinstance(value, list):
        for child in value: yield from blocks(child)
    elif isinstance(value, dict):
        if 'Questions' in value:
            yield value.get('BlockName', 'Unnamed block'), list(questions(value['Questions']))
        else:
            for child in value.values(): yield from blocks(child)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))

def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()

## 1. Participants and data components

| Component | Meaning | Use |
|---|---|---|
| **A** | Earlier questions not repeated in wave 4, with answers | Participant profile input |
| **B** | Earlier answers to the repeated questions | Training labels |
| **C** | Later answers to those repeated questions | Evaluation reference |

There are **2,058 participants**, stored in seven chunks. Our experiment splits **people**, so one person's records cannot cross train/validation/test.

In [2]:
ids = list(map(str, waves.to_table(columns=['pid'])['pid'].to_pylist()))
assert len(ids) == len(set(ids)) == 2058 and len(paths) == 7
# Reconstruct the original frozen split: seed 42; participant 574 reserved for training.
remaining = [pid for pid in sorted(ids) if pid != '574']
random.Random(42).shuffle(remaining)
splits = {'train': ['574'] + remaining[:1439],
          'validation': remaining[1439:1748], 'test': remaining[1748:]}
split_path = Path(os.environ.get('TWIN_SPLIT_PATH', 'data/participant_split.csv'))
if split_path.is_file():
    saved = pd.read_csv(split_path, dtype={'pid': str})
    assert saved.pid.is_unique and set(saved.pid) == set(ids)
    assert set(saved['split']) == set(splits)
    for name in splits:
        assert set(saved.loc[saved['split'] == name, 'pid']) == set(splits[name])
assert '574' in splits['train']
display(pd.DataFrame({'Participants': {'Total': len(ids),
    **{name: len(group) for name, group in splits.items()}}}))

,Participants
Total,2058
train,1440
validation,309
test,309


## 2. Question types

Counts below are **catalogue question records across all branches**, excluding display instructions. They are not per-person answer counts. Matrix rows and slider statements can produce multiple scalar answers.

Multiple-choice and matrix answers may be nominal or ordered. **B's six text-entry questions require numbers**; A also includes free text and multiple-selection questions.

In [3]:
types = pd.DataFrame({name: Counter(q['QuestionType'] for q in catalog
    if q.get('source') == field and q['QuestionType'] != 'DB')
    for name, field in FIELDS.items()}).fillna(0).astype(int)
types = types.reindex(['MC', 'Matrix', 'Slider', 'TE'], fill_value=0)
types.index = ['Multiple choice', 'Matrix', 'Slider', 'Text entry']
types.columns = ['A records', 'B/C records (all branches)']
types.loc['Total'] = types.sum()
assert types.loc['Total'].tolist() == [158, 84]
display(types)

,A records,B/C records (all branches)
Multiple choice,107,68
Matrix,29,7
Slider,0,3
Text entry,22,6
Total,158,84


## 3. Are A and B identical across participants?

**A is mostly shared, not fully identical.** It has 150 exact question definitions common to everyone, but some records are absent and Forward Flow uses different starting words. Each participant supplies their own answers.

**B also contains shared questions**, but complete question configurations differ. Participants receive different branches; pricing presents the same 40 products with different prices and product order. Therefore, the same pricing QuestionID can refer to a different product or price.

The audit below removes answers and compares full question definitions, including block, preceding instructions and option order. For retrieval, match the **actual target condition**, not only its QuestionID.

In [4]:
counts = {name: Counter() for name in FIELDS}
presence = {name: Counter() for name in FIELDS}
configurations = {name: set() for name in FIELDS}
seen = set()
for batch in waves.scanner(columns=['pid', *FIELDS.values()], batch_size=32).to_batches():
    for row in batch.to_pylist():
        pid = str(row.pop('pid'))
        assert pid not in seen
        seen.add(pid)
        for name, field in FIELDS.items():
            metadata = strip_answers(decode(row.pop(field)))
            signatures = []
            for block, items in blocks(metadata):
                preceding = []
                for q in items:
                    if q['QuestionType'] == 'DB':
                        preceding.append(q)
                    else:
                        signatures.append(digest({'BlockName': block,
                            'PrecedingDisplayInstructions': list(preceding), 'Question': q}))
            counts[name][len(signatures)] += 1
            presence[name].update(set(signatures))
            configurations[name].add(digest(sorted(set(signatures))))
assert seen == set(ids)
overlap = pd.DataFrame({name: {
    'Stored question records per person': ' / '.join(map(str, sorted(counts[name]))),
    'Exact definitions common to everyone': sum(n == len(ids) for n in presence[name].values()),
    'Distinct complete question configurations': len(configurations[name])}
    for name in FIELDS})
assert overlap.loc['Exact definitions common to everyone'].tolist() == [150, 6]
assert overlap.loc['Distinct complete question configurations'].tolist() == [20, 2058]
display(overlap)

,A,B
Stored question records per person,156 / 157 / 158,63
Exact definitions common to everyone,150,6
Distinct complete question configurations,20,2058


## 4. The leakage trap: full persona reveals C

The merged **`full_persona` uses the later C answer for repeated questions**. Using it as the profile input can expose the answer being predicted. A summary derived from it is also excluded.

The check below uses **training participant 574** and finds a question where **B differs from C**, while the full persona matches C. This distinguishes later-answer leakage from ordinary repeat agreement.

**Prevention:** use `wave1_3_persona_json` as A; remove answer fields from the target question; train on training participants' B; score against held-out C. The earlier CSV contains **A + B**, so its entire row is not a safe A profile either. Keep participants disjoint across splits.

In [5]:
def read_person(dataset, columns, pid='574'):
    key = int(pid) if pa.types.is_integer(dataset.schema.field('pid').type) else pid
    return dataset.to_table(columns=columns, filter=ds.field('pid') == key).to_pylist()

def question_map(value):
    result = defaultdict(list)
    for q in questions(decode(value)):
        result[(q['QuestionID'], q['QuestionType'])].append(q)
    return result

assert '574' in splits['train']
person, = read_person(waves, [*FIELDS.values(), 'wave4_Q_wave4_A'])
a, b, c = [question_map(person[field]) for field in [*FIELDS.values(), 'wave4_Q_wave4_A']]
full = None
for path in sorted((ROOT/'full_persona').rglob('*.parquet')):
    selected = read_person(ds.dataset(str(path), format='parquet'), ['persona_json'])
    if selected:
        record, = selected
        full = question_map(record['persona_json'])
        break
proof = []
if full is not None:
    for key, earlier in b.items():
        later, merged = c.get(key, []), full.get(key, [])
        if key[1] == 'MC' and len(earlier) == len(later) == len(merged) == 1:
            bv, cv, fv = [items[0].get('Answers') for items in [earlier, later, merged]]
            if bv and cv and canonical(bv) != canonical(cv) and canonical(fv) == canonical(cv):
                assert key not in a
                proof.append({'Question': key[0], 'Earlier B': bv['SelectedByPosition'],
                              'Later C': cv['SelectedByPosition'], 'Full persona': fv['SelectedByPosition']})
    assert proof, 'No discriminating leakage example found; check the dataset snapshot.'
    display(pd.DataFrame(proof[:1]))
    display(Markdown('**Leakage confirmed:** the full persona contains C, not B, for this question.'))
else:
    display(Markdown('**Empirical check not run:** download `full_persona/chunks/persona_chunk_001.parquet` to verify participant 574.'))

,Question,Earlier B,Later C,Full persona
0,QID158,3,4,4


**Leakage confirmed:** the full persona contains C, not B, for this question.